# Парсинг данных ТОП 250-Фильмов

## Этапы парсинга
1. Поиск данных
2. Получение информации
3. Сохранение данных

### Подключение библиотек

In [1]:
from bs4 import BeautifulSoup as bs
import requests
import pandas as pd
import time
import re

### URL ссылка сайта и получение информации

In [2]:
# GET - запрос
url = 'https://www.kinoafisha.info/rating/movies/imdb/'
page = requests.get(url)

In [3]:
page.status_code

200

In [4]:
soup = bs(page.text, 'html.parser')

# Загрузка главной страницы

In [5]:
result_list = {
    'title': [],
    'rating': [],
    'year': [],
    'country': [],
    'description': [],
}

base_url = "https://www.kinoafisha.info"
rating_url = "https://www.kinoafisha.info/rating/movies/imdb/"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

print("Загружаем главную страницу рейтинга...")
try:
    response = requests.get(rating_url, headers=headers, timeout=30)
    response.raise_for_status()
except requests.RequestException as e:
    print(f"Ошибка загрузки: {e}")
    exit()

soup = bs(response.text, 'html.parser')
rating_blocks = soup.find_all(string=lambda text: text and re.match(r'^\s*\d+\.\d+\s*$', text))

print(f"Найдено блоков с рейтингами: {len(rating_blocks)}")

Загружаем главную страницу рейтинга...
Найдено блоков с рейтингами: 260


### Парсинг

In [ ]:
for i, rating_el in enumerate(rating_blocks[:250]):
    try:
        # 1. Рейтинг
        rating = rating_el.strip() if isinstance(rating_el, str) else rating_el.get_text(strip=True)
        
        # 2. Название и ссылка
        link_el = rating_el.find_next('a', href=lambda h: h and '/movies/' in h and '/rating/' not in h)
        
        if not link_el:
            print(f"Пропускаем элемент {i+1}: нет ссылки на фильм")
            continue
            
        title = link_el.get_text(strip=True)
        film_url = link_el.get('href')
        if not film_url.startswith('http'):
            film_url = 'https://www.kinoafisha.info' + film_url
        
        # 3. Мета-инфо (Год, Страна)
        year, country = 'N/A', 'N/A'
        
        parent = link_el.find_parent()
        if parent:
            meta_text = parent.get_text(separator=' ', strip=True)
        
        if not meta_text or len(meta_text) < 5:
            next_sib = link_el.find_next_sibling()
            if next_sib:
                meta_text = next_sib.get_text(strip=True)
            else:
                next_text = rating_el.find_next(string=lambda t: t and re.search(r'(19|20)\d{2}', str(t) if t else ''))
                if next_text:
                    meta_text = next_text

        if meta_text:
            year_match = re.search(r'(19|20)\d{2}', str(meta_text))
            if year_match:
                year = year_match.group(0)
                parts = str(meta_text).split(year)
                
                if len(parts) > 1:
                    country_part = parts[1].strip().strip(',').strip()
                    if country_part and len(country_part) > 2:
                        country = country_part.split(',')[0].split(' ')[0].strip()

        # 4. Описание
        description = 'Описание отсутствует'
        try:
            print(f"  Загружаем страницу {film_url}...")
            film_resp = requests.get(film_url, headers=headers, timeout=10)
            
            if film_resp.status_code == 200:
                film_soup = bs(film_resp.text, 'html.parser')
                
                # Вариант 1: Классический блок с описанием на кинопоиске
                desc_elem = film_soup.find('div', class_=re.compile(r'description|synopsis|plot|film\-description|movie\-description|storyline', re.I))
                
                # Вариант 2: Параграф с большим текстом
                if not desc_elem:
                    for p in film_soup.find_all('p'):
                        p_text = p.get_text(strip=True)
                        if len(p_text) > 150 and any(word in p_text.lower() for word in ['фильм', 'история', 'сюжет', 'главный герой']):
                            desc_elem = p
                            break
                
                # Вариант 3: Элемент с itemprop="description"
                if not desc_elem:
                    desc_elem = film_soup.find(attrs={'itemprop': 'description'})
                
                # Вариант 4: Мета-тег description
                if not desc_elem:
                    meta_desc = film_soup.find('meta', attrs={'name': 'description'})
                    if meta_desc:
                        description = meta_desc.get('content', 'Описание отсутствует')
                        print(f"    Найдено в meta description: {description[:50]}...")
                    else:
                        # Вариант 5: Любой текст между <p> в блоке контента
                        content_div = film_soup.find('div', class_=re.compile(r'content|article|text', re.I))
                        if content_div:
                            for p in content_div.find_all('p'):
                                p_text = p.get_text(strip=True)
                                if len(p_text) > 100:
                                    desc_elem = p
                                    break
                
                if desc_elem:
                    description = desc_elem.get_text(strip=True)
                    description = re.sub(r'\s+', ' ', description)
                else:
                    print(f"    Не найдено описание на странице")
            
        except requests.exceptions.Timeout:
            print(f"  Таймаут при загрузке {title}")
            description = 'Ошибка загрузки (таймаут)'
        except requests.exceptions.RequestException as e:
            print(f"  Ошибка сети для {title}: {e}")
            description = f'Ошибка загрузки'
        except Exception as e:
            print(f"  Неожиданная ошибка для {title}: {e}")
            description = 'Описание отсутствует'
        
        # 5. Сохранение
        result_list['title'].append(title)
        result_list['rating'].append(rating)
        result_list['year'].append(year)
        result_list['country'].append(country)
        result_list['description'].append(description)
        
    except Exception as e:
        print(f"Ошибка обработки фильма #{i+1}: {e}")
        continue

  Загружаем страницу https://www.kinoafisha.info/movies/7731571/...
  Загружаем страницу https://www.kinoafisha.info/movies/3974172/...
  Загружаем страницу https://www.kinoafisha.info/movies/3229744/...
  Загружаем страницу https://www.kinoafisha.info/movies/4158964/...
  Загружаем страницу https://www.kinoafisha.info/movies/3159600/...
  Загружаем страницу https://www.kinoafisha.info/movies/5475/...
  Загружаем страницу https://www.kinoafisha.info/movies/3725038/...
  Загружаем страницу https://www.kinoafisha.info/movies/3337323/...
  Загружаем страницу https://www.kinoafisha.info/movies/4366576/...
  Загружаем страницу https://www.kinoafisha.info/movies/8125647/...
  Загружаем страницу https://www.kinoafisha.info/movies/5227/...
  Загружаем страницу https://www.kinoafisha.info/movies/4162346/...
  Загружаем страницу https://www.kinoafisha.info/movies/3963633/...
  Загружаем страницу https://www.kinoafisha.info/movies/7794008/...
  Загружаем страницу https://www.kinoafisha.info/movie

In [ ]:
df = pd.DataFrame(result_list)

### Вывод таблицы

In [ ]:
df.head(10)

### Сохранение в csv файл

In [ ]:
df.to_csv('top250.csv', index=False, encoding='utf-8-sig')

### Парсинг с помощью API к кинопоиску

In [ ]:
API_TOKEN = "QKWZN69-DNS4ZWV-HDG2822-RWXZH9Q"
headers = {'X-API-KEY': API_TOKEN, 'Content-Type': 'application/json'}
result_list = {'title': [], 'year': [], 'country': [], 'rating': [], 'description': []}

In [ ]:
page = 1
collected = 0
max_attempts = 10
attempts = 0

while collected < 250 and attempts < max_attempts:
    try:
        url = f"https://api.kinopoisk.dev/v1.4/movie?lists=top250&limit=50&page={page}"
        
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code != 200:
            print(f"Ошибка API: статус {response.status_code}")
            attempts += 1
            time.sleep(2)
            continue
            
        data = response.json()
        movies = data.get('docs', [])
        
        for movie in movies:
            if collected >= 250:
                break
            
            # Название
            title = movie.get('name') or movie.get('alternativeName', 'Не указано')
            
            # Год
            year = movie.get('year', 'Не указан')
            
            # Страна
            countries = movie.get('countries', [])
            country = countries[0].get('name', 'Не указана') if countries else 'Не указана'
            
            # Рейтинг
            rating_data = movie.get('rating', {})
            rating = rating_data.get('kp', None)
            if rating:
                rating = str(round(float(rating), 1))
            else:
                rating = 'Нет рейтинга'
            
            # Описание
            description = movie.get('description') or movie.get('description', 'Описание отсутствует')
            
            result_list['title'].append(title)
            result_list['year'].append(year)
            result_list['country'].append(country)
            result_list['rating'].append(rating)
            result_list['description'].append(description)
            
            collected += 1
            print(f"{collected}. {title[:30]} ({year}) - {country}")
        
        page += 1
        attempts = 0
        time.sleep(0.5)
        
    except requests.exceptions.Timeout:
        print(f"Таймаут на странице {page}, пробуем снова...")
        attempts += 1
        time.sleep(3)
    except requests.exceptions.RequestException as e:
        print(f"Сетевая ошибка: {e}")
        attempts += 1
        time.sleep(3)
    except Exception as e:
        print(f"Неожиданная ошибка: {e}")
        attempts += 1
        time.sleep(2)

print(f"\nСобрано {collected} фильмов из 250")

# Создание DataFrame
df_new = pd.DataFrame(result_list)

In [ ]:
# Выводим первые 10 для проверки
print(df_new[['title', 'year', 'country', 'rating']].head(10).to_string(index=False))

### Сохраняем датасет в CSV

In [ ]:
file_name = 'top250.csv'

try:
    try:
        df_existing = pd.read_csv(file_name, encoding='utf-8-sig')
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        
        df_combined.to_csv(file_name, index=False, encoding='utf-8-sig')
        print(f"Данные успешно сохранены в файл")
        
    except FileNotFoundError:
        df_new.to_csv(file_name, index=False, encoding='utf-8-sig')
        print(f"Создан новый файл '{file_name}' с {len(df_new)} фильмами")
        
except Exception as e:
    print(f"Ошибка при сохранении: {e}")
    
finally:
    try:
        df_result = pd.read_csv(file_name, encoding='utf-8-sig')
        print(f"Файл '{file_name}' содержит {len(df_result)} записей")
        print("\nПоследние 5 записей в файле:")
        print(df_result[['title', 'year', 'country', 'rating']].tail(5).to_string(index=False))
    except:
        print("Не удалось проверить итоговый файл")